# 04 – Subvraag weer: waar en bij wie heeft regen de meeste impact?
**Subvraag (Jesse):** *Verschilt het effect van regen per station, en verliezen stations dicht bij een
metrostation meer ritten als het regent?*

**Idee:** als mensen bij regen de fiets laten staan en de metro nemen, dan verwacht je dat stations dicht bij
de metro **sterker** dalen dan stations ver weg. Let op: we hebben geen metrogebruik per dag (Kaggle = jaartotalen),
dus dit is een *mogelijke verklaring*, geen bewijs.

**Methode (zelfde als notebook 03, maar per station):**
voor elk station vergelijken we het aantal vertrekken op regendagen met het gemiddelde van droge dagen in
**dezelfde maand** en van **hetzelfde type dag** (werkdag/weekend). Zo corrigeren we voor seizoen en weekend.

De rekenfuncties staan in `weather_impact.py`, zodat het Streamlit-dashboard **exact dezelfde** berekening gebruikt.

**Nodig:** notebook 01 en 02 gedraaid. Van Jurre (optioneel): `data/processed/station_metro_distance.parquet`
met kolommen `bikepoint_id`, `metro_distance_m` en eventueel `metro_group`. Zonder dat bestand gebruiken we tijdelijk
de afstand tot het centrum (Bank).

In [ ]:
# Load the prepared tables and the shared analysis functions
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import weather_impact as wi

daily = pd.read_parquet(wi.PROC / "daily.parquet")
daily_station = pd.read_parquet(wi.PROC / "daily_station.parquet")
stations = pd.read_parquet(wi.PROC / "stations.parquet")
hourly = pd.read_parquet(wi.PROC / "hourly.parquet")
print(f"{len(daily)} dagen, {daily_station['start_bp'].nunique()} stations, wind-kolom: {wi.wind_column(daily)}")

## 1. Tabel station × dag (inclusief dagen met 0 vertrekken)
`daily_station` bevat alleen dagen waarop een station **wel** ritten had. Een rustig station dat op een regendag
0 vertrekken heeft, zou dan ontbreken, en dat onderschat juist het regeneffect. Daarom vullen we aan met nullen,
maar alleen tussen de eerste en laatste rit van dat station (nieuwe of opgeheven stations).

In [ ]:
# Build the full station x day table (missing days = 0 departures) and attach the weather
sd = wi.build_station_day(daily_station, daily, stations)
sd.to_parquet(wi.PROC / "station_day.parquet", index=False)
print(f"{len(sd):,} rijen (station × dag), waarvan {(sd['departures'] == 0).mean():.1%} met 0 vertrekken")
sd.head()

## 2. Regengevoeligheid per station
- **Regendag** = neerslag ≥ `THRESHOLD` mm (in het dashboard instelbaar met een slider).
- **Droog** = < 0,2 mm. Dagen daartussen tellen niet mee.
- We nemen alleen stations mee met genoeg ritten (≥ 5 vertrekken per droge dag) en genoeg regendagen (≥ 10).
  Bij kleine aantallen is een percentage te onbetrouwbaar (van 2 naar 1 rit = −50%).

In [ ]:
# Compute the rain effect per station and for the whole city
THRESHOLD = 1.0
effect = wi.rain_effect_per_station(sd, stations, threshold=THRESHOLD)
city_pct, n_rain_days = wi.citywide_effect(sd, threshold=THRESHOLD)
print(f"Heel Londen: {city_pct:+.1f}% vertrekken op regendagen (≥{THRESHOLD} mm, n={n_rain_days} dagen)")
print(f"{len(effect)} stations geanalyseerd")
if effect.empty:
    print("⚠️ Geen stations over: verlaag min_per_day of min_rain_days in rain_effect_per_station(...)")
effect["effect_pct"].describe().round(1)

In [ ]:
# Distribution of the per-station effect: is it the same everywhere or are there big differences?
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(effect["effect_pct"], bins=40, color="#4292c6")
ax.axvline(city_pct, color="black", ls="--", label=f"heel Londen ({city_pct:+.0f}%)")
ax.set_xlabel("% vertrekken op regendag t.o.v. droge dag"); ax.set_ylabel("aantal stations")
ax.set_title("Regen raakt niet elk station even hard"); ax.legend()
plt.show()

In [ ]:
# Stations that are the most and least sensitive to rain
cols = ["name", "dry_departures_per_day", "rain_days", "effect_pct"]
print("Meest gevoelig voor regen:")
print(effect.nsmallest(10, "effect_pct")[cols].round(1).to_string(index=False))
print("\nMinst gevoelig voor regen:")
print(effect.nlargest(10, "effect_pct")[cols].round(1).to_string(index=False))

## 3. Kaart: waar is Londen het gevoeligst voor regen?

In [ ]:
# Simple map: colour = rain effect, size = number of rides (the dashboard has an interactive version)
fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(effect["lon"], effect["lat"], c=effect["effect_pct"], cmap="RdBu",
                vmin=-60, vmax=20, s=np.sqrt(effect["dry_departures_per_day"]) * 8, alpha=.8, edgecolor="grey", lw=.3)
plt.colorbar(sc, label="% vertrekken op regendag")
ax.set_title("Regengevoeligheid per station (rood = groot verlies)")
ax.set_xlabel("lon"); ax.set_ylabel("lat")
plt.show()

## 4. Koppeling met de metro (deel van Jurre)
Zodra Jurre zijn bestand aanlevert, worden zijn afstandsgroepen automatisch gebruikt.
Tot die tijd tonen we de afstand tot het centrum, zodat de grafiek alvast werkt.

In [ ]:
# Attach the distance groups (Jurre's metro distance, or distance to the centre as a placeholder)
effect_d, source = wi.add_metro_distance(effect, stations)
label = "afstand tot metrostation" if source == "metro" else "afstand tot centrum (TIJDELIJK, wacht op Jurre)"
by_group = effect_d.groupby("metro_group", observed=True).agg(
    stations=("bikepoint_id", "size"),
    mean_effect=("effect_pct", "mean"),
    weighted_effect=("effect_pct", lambda x: np.average(x, weights=effect_d.loc[x.index, "dry_departures_per_day"])))
print("Gegroepeerd op:", label)
by_group.round(1)

In [ ]:
# Bar chart per distance group
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(by_group.index.astype(str), by_group["weighted_effect"], color="#08519c")
ax.axhline(0, color="black", lw=.8)
for i, (v, n) in enumerate(zip(by_group["weighted_effect"], by_group["stations"])):
    ax.text(i, v, f"{v:+.0f}%\n(n={n})", ha="center", va="top" if v < 0 else "bottom")
ax.set_ylabel("% vertrekken op regendag\n(gewogen naar drukte)")
ax.set_xlabel(label)
ax.set_title("Verliezen stations dicht bij de metro meer ritten bij regen?")  # ← fill in conclusion
plt.show()

In [ ]:
# Is the relationship significant? Spearman correlation between distance and rain effect
dist_col = "metro_distance_m" if "metro_distance_m" in effect_d.columns else "centre_distance_km"
rho, p = stats.spearmanr(effect_d[dist_col], effect_d["effect_pct"], nan_policy="omit")
print(f"Spearman ρ tussen {dist_col} en regeneffect: {rho:+.2f} (p = {p:.3g})")
print("Positief ρ = hoe verder weg, hoe minder verlies (past bij de metro-hypothese).")

## 5. Woon-werk of recreatie? Uurpatroon op droge en regendagen
Als vooral de spitsuren overeind blijven bij regen, zijn het de recreatieve ritten die wegvallen.

In [ ]:
# Average rides per hour on dry vs. rainy days (working days only)
h = hourly[hourly["day_type"] == "werkdag"].dropna(subset=["precip_mm"]).copy()
h["weather"] = np.select([h["precip_mm"] < wi.DRY_MM, h["precip_mm"] >= THRESHOLD], ["droog", "regen"], "tussenin")
prof = h[h["weather"] != "tussenin"].groupby(["weather", "hour"])["rides"].mean().unstack(0)
prof["verschil_pct"] = (prof["regen"] / prof["droog"] - 1) * 100

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
prof[["droog", "regen"]].plot(ax=ax[0]); ax[0].set_title("Ritten per uur (werkdagen)"); ax[0].set_xlabel("uur")
ax[1].bar(prof.index, prof["verschil_pct"], color="#4292c6"); ax[1].axhline(0, color="black", lw=.8)
ax[1].set_title("% verschil regen vs. droog per uur"); ax[1].set_xlabel("uur")
plt.show()

## 6. Voorspelmodel voor de wat-als-tool
Lineaire regressie op `log(rides)` met neerslag, temperatuur, wind (als die er is), weekdag, maand en feestdag.
Dit model gebruikt het dashboard voor de sliders.

In [ ]:
# Fit the model and show the effect of each weather factor
model, wind_col = wi.fit_daily_model(daily)
print(f"R² = {model.rsquared:.2f}  |  n = {int(model.nobs)} dagen")
for var, unit in [("precip_mm", "mm regen"), ("temp_mean", "°C warmer"), (wind_col, "km/u meer wind")]:
    if var:
        b = model.params[var]; lo, hi = model.conf_int().loc[var]
        print(f"Per {unit:<15} {(np.exp(b)-1)*100:+.1f}%  (95%-BI {(np.exp(lo)-1)*100:+.1f}% t/m {(np.exp(hi)-1)*100:+.1f}%)")

In [ ]:
# Example what-if: a Tuesday in October with and without rain
for mm in [0, 2, 8, 15]:
    est, lo, hi = wi.predict_rides(model, precip_mm=mm, temp_mean=12, weekday=1, month=10,
                                   wind=15, wind_col=wind_col)
    print(f"{mm:>2} mm regen → ±{est:,.0f} ritten  (95%-interval {lo:,.0f} – {hi:,.0f})")

In [ ]:
# Check: how well does the model predict? Predicted vs. actual per day
d = daily.dropna(subset=["precip_mm", "temp_mean"]).copy()
if wind_col:
    d[wind_col] = d[wind_col].fillna(d[wind_col].median())
d["predicted"] = np.exp(model.predict(d)) * np.mean(np.exp(model.resid))
mae = (d["predicted"] - d["rides"]).abs().mean()
fig, ax = plt.subplots(figsize=(13, 3.5))
ax.plot(d["date"], d["rides"], lw=.8, label="werkelijk"); ax.plot(d["date"], d["predicted"], lw=.8, label="voorspeld")
ax.set_title(f"Model vs. werkelijkheid (gem. afwijking {mae:,.0f} ritten per dag)"); ax.legend()
plt.show()

## Conclusie subvraag (concept)
- Effect van regen voor heel Londen: …
- Verschil tussen stations: …
- Metro-afstand: stations dicht bij de metro verliezen wel/niet meer ritten bij regen: …
- Uurpatroon: vooral spits/recreatie valt weg: …
- Kanttekening: geen dagelijkse metrodata, dus overstappen op de metro is een mogelijke verklaring, geen bewijs.